<a href="https://colab.research.google.com/github/NikitaIvagin/ml-portfolio/blob/main/tabular_data/car_price_prediction/Car_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

%matplotlib inline
url = "https://storage.yandexcloud.net/academy.ai/japan_cars_dataset.csv"
df = pd.read_csv(url)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2318 entries, 0 to 2317
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       2318 non-null   int64 
 1   price            2318 non-null   int64 
 2   mark             2318 non-null   object
 3   model            2318 non-null   object
 4   year             2318 non-null   int64 
 5   mileage          2318 non-null   int64 
 6   engine_capacity  2318 non-null   int64 
 7   transmission     2318 non-null   object
 8   drive            2318 non-null   object
 9   hand_drive       2318 non-null   object
 10  fuel             2318 non-null   object
dtypes: int64(5), object(6)
memory usage: 199.3+ KB


In [ ]:
df.head()

,Unnamed: 0,price,mark,model,year,mileage,engine_capacity,transmission,drive,hand_drive,fuel
0,0,80,nissan,march,2003,80000,1240,at,2wd,rhd,gasoline
1,1,110,nissan,march,2010,53000,1200,at,2wd,rhd,gasoline
2,2,165,nissan,lafesta,2005,47690,2000,at,2wd,rhd,gasoline
3,3,190,toyota,avensis,2008,130661,1990,at,2wd,rhd,gasoline
4,4,190,daihatsu,mira,2006,66300,660,at,2wd,rhd,gasoline


In [ ]:
iso = IsolationForest(contamination=0.1)
outliers = iso.fit_predict(df.select_dtypes(include=np.number))
df = df[outliers == 1]

# Логарифмирование целевой переменной
df['log_price'] = np.log1p(df['price'])

# Фильтрация по пробегу
df = df[df['mileage'] < df['mileage'].quantile(0.99)]

In [ ]:
X = df.drop(['price', 'log_price'], axis=1)
y = df['log_price']

# Разделение на числовые и категориальные признаки
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# One-hot encoding для категориальных
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [ ]:
scaler = RobustScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
model = tf.keras.Sequential([
        Dense(512, input_shape=(X_train.shape[1],),
              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        BatchNormalization(),
        Activation('swish'),
        Dropout(0.3),

        Dense(256),
        BatchNormalization(),
        Activation('swish'),
        Dropout(0.2),

        Dense(128),
        BatchNormalization(),
        Activation('swish'),

        Dense(64),
        BatchNormalization(),
        Activation('swish'),

        Dense(1)
    ])

model.compile(
        optimizer=Adam(learning_rate=0.005),
        loss='mse',
        metrics=['mae']
    )

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
callbacks = [
    EarlyStopping(patience=100, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=10)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=500,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/500
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 84ms/step - loss: 32.5225 - mae: 5.5198 - val_loss: 14.1127 - val_mae: 3.6872 - learning_rate: 0.0050
Epoch 2/500
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.8552 - mae: 0.7263 - val_loss: 3.3158 - val_mae: 1.6698 - learning_rate: 0.0050
Epoch 3/500
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2871 - mae: 0.3829 - val_loss: 1.2965 - val_mae: 1.0634 - learning_rate: 0.0050
Epoch 4/500
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2662 - mae: 0.3598 - val_loss: 0.3622 - val_mae: 0.4925 - learning_rate: 0.0050
Epoch 5/500
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1956 - mae: 0.2962 - val_loss: 0.1381 - val_mae: 0.2453 - learning_rate: 0.0050
Epoch 6/500
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1411 - mae: 0.2398 - val_loss: 0.1062 - val_mae: 0.1758 - learning_rate: 0.0050
Epoch 7/500
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1015 - mae: 0.1910 - val_loss: 0.1254 - val_mae: 0.2353 - learning_rate: 0.0050
Epoch 8/5

In [ ]:
val_pred = np.expm1(model.predict(X_val)).flatten()
val_true = np.expm1(y_val)
val_rmse = np.sqrt(np.mean((val_pred - val_true)**2))
val_error = (val_rmse / val_true.mean()) * 100

test_pred = np.expm1(model.predict(X_test)).flatten()
test_true = np.expm1(y_test)
test_rmse = np.sqrt(np.mean((test_pred - test_true)**2))
test_error = (test_rmse / test_true.mean()) * 100

print(f"\nПроверочная выборка:")
print(f"RMSE: {val_rmse:.2f} | Ошибка: {val_error:.2f}%")
print(f"\nТестовая выборка:")
print(f"RMSE: {test_rmse:.2f} | Ошибка: {test_error:.2f}%")

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 

Проверочная выборка:
RMSE: 20.89 | Ошибка: 2.18%

Тестовая выборка:
RMSE: 17.20 | Ошибка: 1.74%
